# Prompt 工程：多轮对话（OrderBot）

从 `notebookes/13_prompt_engineer_chat.ipynb` 抽取的系统提示与多轮调试逻辑。原笔记用 Panel 做界面，在 VS Code 中常需 `jupyter_bokeh`。此处改为**纯消息列表 + 函数封装**，自上而下执行即可完成多轮演示，无需 GUI依赖。


In [6]:
import os

from llm_config import build_chat_openai

provider = "lmstudio" if os.environ.get("USE_LM_STUDIO", "1") == "1" else "openai"
llm = build_chat_openai(provider=provider, temperature=0)


In [7]:
ORDERBOT_SYSTEM = """
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
""".strip()


In [3]:
def chat_turn(context: list, user_text: str) -> str:
    """追加用户消息，调用 LLM，再把助手回复写回 context（与 OpenAI chat 格式一致）。"""
    context.append({"role": "user", "content": user_text})
    response = llm.invoke(context)
    reply = response.content
    context.append({"role": "assistant", "content": reply})
    return reply


def new_orderbot_session() -> list:
    return [{"role": "system", "content": ORDERBOT_SYSTEM}]


In [4]:
# 固定多轮脚本：无需输入，适合一键跑通 prompt 行为
ctx = new_orderbot_session()
demo_turns = [
    "Hi",
    "I'd like a large cheese pizza and a coke",
    "Delivery to 123 Main St, thanks",
]

for user_line in demo_turns:
    print("User:", user_line)
    assistant = chat_turn(ctx, user_line)
    print("Assistant:", assistant)
    print("---")


User: Hi


TypeError: Completions.create() got an unexpected keyword argument 'provider'

## 可选：交互式输入

在支持 `input()` 的环境中，可取消下面单元格注释，自行多轮试 prompt。


In [5]:
# ctx = new_orderbot_session()
# while True:
#     line = input("You: ").strip()
#     if not line:
#         break
#     print("Assistant:", chat_turn(ctx, line))
